# 00 · Setup & cek data

**Tujuan.** Memastikan lingkungan dan data siap, lengkap, dan selaras dengan kontrak lomba *sebelum*
analisis apa pun. Kalau notebook ini gagal, notebook lain tidak perlu dijalankan.

| | |
|---|---|
| **Input** | `tactical/`, `broadcast/`, repo resmi di `external/more-than-a-hack-2027` |
| **Output** | `cache/reports/00_setup_cek_data.txt` dan `.json` (hasil cek + sidik jari data) |
| **Waktu** | ± 1–2 menit (ATD dibaca secara streaming, file data di-hash) |
| **Privasi** | hanya mencetak angka agregat (jumlah, persentase, lulus/gagal) |

**Yang dicek**

1. Lingkungan: versi Python & pustaka sesuai `requirements.txt`, kernel memakai `.venv`
2. Inventaris aset & sidik jari (SHA-256)
3. Metadata ATD: tim kandang/tandang, slot per tim, arah serang
4. ATD tactical: jumlah baris, urutan `frame_id`, jumlah slot, kehadiran bola
5. Homography tactical & broadcast
6. Smart Tagging: jumlah, kode, tim, jendela kick-off
7. ATD broadcast
8. Scorer resmi & GT publik (uji asap)
9. Ringkasan lulus/gagal

Angka acuan diambil dari DATA.md / FAQ.md. Selisih berarti data kita **berbeda** dari yang dipakai panitia, dan
itu harus diselesaikan dulu (jangan dikoreksi diam-diam).

Setiap cek punya tingkat: **kritis** (gagal = berhenti) atau **peringatan** (dicatat, tidak menghentikan).

In [ ]:
%run ./common.ipynb

## Parameter

Hashing video broadcast (± 14 GB) butuh beberapa menit. Ukuran file sudah cukup sebagai sidik jari, jadi
video tidak di-hash kecuali diminta.

In [ ]:
HASH_DATA = True     # SHA-256 semua file data selain video (± 400 MB, beberapa detik)
HASH_VIDEO = False   # True = hash video juga (lama)

### Pencatat hasil cek

`check(nama, lulus, keterangan, kritis)` mencatat satu baris ke laporan dan ke tabel ringkasan di bagian 9.

In [ ]:
rep = Report("00_setup_cek_data")
CHECKS = []


def check(name, ok, detail="", critical=True):
    ok = bool(ok)
    status = "PASS" if ok else ("FAIL" if critical else "WARN")
    CHECKS.append({"cek": name, "status": status, "kritis": critical, "keterangan": detail})
    rep("[{}] {}{}".format(status, name, " — " + detail if detail else ""))
    return ok

## 1. Lingkungan

Reproduksi **persis** butuh versi pustaka yang sama. Versi terpasang dibandingkan dengan pin di
`requirements.txt`. Di sini selisih hanya **peringatan** (mungkin notebook dibuka dengan Jupyter global), tetapi
run final untuk submission **wajib** lewat `run_all` di `.venv`.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

pins = {}
for line in (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        name, ver = line.split("==")
        pins[name.strip()] = ver.strip()

rows = []
for name, want in pins.items():
    try:
        have = version(name)
    except PackageNotFoundError:
        have = "-"
    rows.append({"paket": name, "dipin": want, "terpasang": have, "sama": have == want})
env = pd.DataFrame(rows).set_index("paket")
rep.table(env, "versi pustaka:")

in_venv = (ROOT / ".venv") in Path(sys.executable).resolve().parents
check("Python 3.11", sys.version_info[:2] == (3, 11), sys.version.split()[0], critical=False)
check("kernel memakai .venv proyek", in_venv, "" if in_venv else "kernel = Python lain", critical=False)
check("versi pustaka = requirements.txt", env.sama.all(),
      "" if env.sama.all() else "beda: " + ", ".join(env.index[~env.sama]), critical=False)

## 2. Inventaris aset & sidik jari

Setiap aset dicari dengan `asset_path()`. Sidik jari SHA-256 disimpan di laporan JSON: kalau suatu hari skor
berubah tanpa perubahan kode, hal pertama yang dicek adalah apakah datanya sama.

Video tidak wajib: seluruh pipeline Fase 1–3 berbasis ATD, Smart Tagging, dan homography. Video tactical
hanya dibutuhkan untuk stretch goal 3.6 (deteksi bola dari video), jadi ketiadaannya hanya peringatan.

In [ ]:
inventory, fingerprint = [], {}
for source in ("tactical", "broadcast"):
    for name in games.SOURCES[source]["assets"]:
        path = asset_path(name, source)
        inventory.append({"sumber": source, "aset": name, "ada": path is not None,
                          "file": path.name if path else "-",
                          "MB": round(path.stat().st_size / 2 ** 20, 1) if path else np.nan})
        if path is None:
            continue
        is_video = name == "video"
        if (HASH_DATA and not is_video) or (HASH_VIDEO and is_video):
            with timer("sha256 " + path.name):
                fingerprint[path.name] = sha256(path)
        else:
            fingerprint[path.name] = "bytes:{}".format(path.stat().st_size)

inv = pd.DataFrame(inventory).set_index(["sumber", "aset"])
rep.table(inv, "aset data:")

non_video = inv.drop(index="video", level="aset")
check("semua aset non-video ada", non_video.ada.all(),
      "" if non_video.ada.all() else "hilang: " + ", ".join("/".join(i) for i in non_video.index[~non_video.ada]))
check("video tactical ada (hanya untuk stretch 3.6)", inv.loc[("tactical", "video"), "ada"], critical=False)
check("video broadcast ada (opsional)", inv.loc[("broadcast", "video"), "ada"], critical=False)

## 3. Metadata ATD (tactical)

File XML FIFA/EPTS memetakan setiap **slot track** (46 buah, bukan pemain!) ke sebuah tim. Dua hal harus benar
sebelum fitur apa pun dibuat:

1. **Tim mana yang kandang (`H`).** Dua cara membaca: `idLocalTeam` pada elemen `Score`, dan konvensi
   baseline resmi (tim pertama = kandang). Keduanya harus sepakat, dan kandang harus Barcelona (DATA.md).
2. **Urutan slot.** Grup ke-*i* di setiap baris ATD adalah slot ke-*i*. Urutan elemen `Player` (dipakai
   baseline) harus sama dengan urutan kanal `player<id>_x` di definisi data.

Arah serang babak 1 (`attack_direction_first_half`) dicatat untuk fitur arah serang nanti.

In [ ]:
def _local(tag):
    return tag.split("}")[-1]


def _child_text(el, *names):
    for child in el:
        if _local(child.tag).lower() in names:
            return (child.text or "").strip()
    return ""


meta_path = require_asset("atd_meta")
meta_root = ET.parse(meta_path).getroot()
team_els = [el for el in meta_root.iter() if _local(el.tag) == "Team" and el.get("id")]
team_name = {t.get("id"): _child_text(t, "name", "n") for t in team_els}
local_ids = [el.get("idLocalTeam") for el in meta_root.iter()
             if _local(el.tag) == "Score" and el.get("idLocalTeam")]
player_els = [el for el in meta_root.iter() if _local(el.tag) == "Player" and el.get("id") and el.get("teamId")]
player_order = [int(el.get("id")) for el in player_els]
channel_order = [int(i) for i in re.findall(r'playerChannelId="player(\d+)_x"',
                                            meta_path.read_text(encoding="utf-8"))]

home_id = team_els[0].get("id") if team_els else None
away_id = team_els[1].get("id") if len(team_els) > 1 else None
slot_is_home = np.array([el.get("teamId") == home_id for el in player_els])
N_SLOTS = len(player_els)

attack_h1 = None
for el in team_els[0].iter() if team_els else []:
    if _local(el.tag) == "ProviderParameter" and _child_text(el, "name", "n") == "attack_direction_first_half":
        attack_h1 = _child_text(el, "value")

rep("tim: H = {} | A = {}".format(team_name.get(home_id), team_name.get(away_id)))
rep("slot: total {} | H {} | A {}".format(N_SLOTS, int(slot_is_home.sum()), int((~slot_is_home).sum())))
rep("arah serang H babak 1: {}".format(attack_h1))

check("XML punya tepat 2 tim", len(team_els) == 2, str(len(team_els)))
check("tim pertama = idLocalTeam", bool(local_ids) and local_ids[0] == home_id,
      "idLocalTeam tidak ada" if not local_ids else "")
check("kandang = Barcelona", "barcelona" in (team_name.get(home_id) or "").lower(), team_name.get(home_id))
check("46 slot track", N_SLOTS == 46, str(N_SLOTS))
check("urutan slot Player = urutan kanal data", player_order == channel_order,
      "{} vs {} entri".format(len(player_order), len(channel_order)))
check("arah serang babak 1 terbaca", attack_h1 in ("left_to_right", "right_to_left"), str(attack_h1), critical=False)

## 4. ATD tactical

Format satu baris: `frame_id:slot1;slot2;…;slotN:bola`, tiap grup `x,y,kecepatan` dalam koordinat 0–1, `NaN` =
tidak terlihat. Kontrak (DATA.md):

| Cek | Acuan |
|---|---|
| jumlah baris | 199.319 |
| `frame_id` | 1…199.319 berurutan → `frame_video = frame_id − 1` |
| grup per baris | = jumlah slot di XML (46) |
| bola ada, semua frame | 113.488 (56,9%) |
| bola ada, frame dinilai | 94.214 (59,3%) |

Hanya `frame_id`, jumlah grup, dan ada/tidaknya bola yang dibaca — parsing penuh dilakukan di `01_preprocess`.

In [ ]:
def scan_atd(path):
    """Baca ringan ATD: frame_id, himpunan jumlah grup per baris, dan bola ada/tidak per baris."""
    ids, n_groups, ball = [], set(), []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            frame, players, b = line.rstrip("\r\n").split(":")
            ids.append(int(frame))
            n_groups.add(players.count(";") + 1)
            ball.append(not b.startswith("N"))      # "NaN,NaN,..." = bola hilang
    return np.array(ids), n_groups, np.array(ball)


with timer("scan ATD tactical"):
    atd_ids, atd_groups, atd_ball = scan_atd(require_asset("atd_raw"))

n_ball_all, n_ball_graded = int(atd_ball.sum()), int(atd_ball[GRADED].sum()) if len(atd_ball) == N_FRAMES else -1
rep("baris {} | frame_id {}..{} | grup per baris {}".format(len(atd_ids), atd_ids[0], atd_ids[-1], sorted(atd_groups)))
rep("bola ada: semua frame {} ({:.1f}%) | dinilai {} ({:.1f}%)".format(
    n_ball_all, 100 * n_ball_all / len(atd_ball), n_ball_graded, 100 * n_ball_graded / N_GRADED))

check("ATD tactical 199.319 baris", len(atd_ids) == N_FRAMES, str(len(atd_ids)))
check("frame_id = 1..199319 berurutan", len(atd_ids) == N_FRAMES and (atd_ids == np.arange(1, N_FRAMES + 1)).all())
check("grup per baris = slot XML", atd_groups == {N_SLOTS}, str(sorted(atd_groups)))
check("bola ada di 113.488 frame (DATA.md)", n_ball_all == 113488, str(n_ball_all))
check("bola ada di 94.214 frame dinilai (DATA.md)", n_ball_graded == 94214, str(n_ball_graded))

## 5. Homography

Satu matriks 3×3 per frame (pitch 0–1 → piksel). Kolom `valid` = matriks ada; `reprojection_error` = seberapa
bisa dipercaya (makin kecil makin baik).

| | baris | `frame` | valid | catatan |
|---|---|---|---|---|
| tactical | 199.319 | 0…199.318 = baris submission | 100% | median error ± 0,24 saat main |
| broadcast | 164.488 | timeline sendiri | 65,8% | sisanya replay, close-up, penonton |

In [ ]:
def homography_summary(source, n_expected, mask=None):
    df = pd.read_csv(require_asset("homography", source), usecols=["frame", "reprojection_error", "valid"])
    ordered = len(df) == n_expected and (df.frame.to_numpy() == np.arange(n_expected)).all()
    err = df.reprojection_error.to_numpy()
    return {"baris": len(df), "frame_berurutan": bool(ordered),
            "valid_%": round(100 * df.valid.mean(), 2),
            "median_error": round(float(np.nanmedian(err if mask is None else err[mask])), 3)}


homo = {"tactical": homography_summary("tactical", N_FRAMES, GRADED),
        "broadcast": homography_summary("broadcast", N_FRAMES_BROADCAST)}
rep.table(pd.DataFrame(homo).T, "homography (median error tactical = frame dinilai):")

check("homography tactical 199.319 baris, frame berurutan",
      homo["tactical"]["baris"] == N_FRAMES and homo["tactical"]["frame_berurutan"])
check("homography tactical valid 100%", homo["tactical"]["valid_%"] == 100.0, str(homo["tactical"]["valid_%"]))
check("homography broadcast 164.488 baris", homo["broadcast"]["baris"] == N_FRAMES_BROADCAST,
      str(homo["broadcast"]["baris"]), critical=False)
check("homography broadcast valid ≈ 65,8%", abs(homo["broadcast"]["valid_%"] - 65.8) < 0.1,
      str(homo["broadcast"]["valid_%"]), critical=False)

## 6. Smart Tagging

SportsCode XML: setiap `instance` punya `code`, `start`/`end` (detik sejak frame pertama video tactical), dan
label `group`/`text`. Konversi ke frame video: `round(detik × 25)` (DATA.md, sudah diverifikasi panitia).

| Cek | Acuan |
|---|---|
| jumlah instans | 638 |
| `Team` | Barcelona 314 · Atletic Club 288 · N/A 36 |
| `SET PIECES` | 59 |
| nama tim di tag cocok dengan nama tim di ATD | tidak ada nilai `Team` yang tak dikenal |
| jendela `Kick Off (Start)` tiap babak memuat kick-off sebenarnya | 11.185 dan 113.742 |

In [ ]:
tag_rows = []
for ins in ET.parse(require_asset("smart_tagging")).getroot().iter("instance"):
    row = {"code": ins.findtext("code"), "start_s": float(ins.findtext("start")), "end_s": float(ins.findtext("end"))}
    for lab in ins.findall("label"):
        row[lab.findtext("group")] = lab.findtext("text")
    tag_rows.append(row)
tags = pd.DataFrame(tag_rows)
tags["start_f"] = (tags.start_s * FPS).round().astype(int)
tags["end_f"] = (tags.end_s * FPS).round().astype(int)

names = {(team_name.get(home_id) or "").strip().lower(): "H", (team_name.get(away_id) or "").strip().lower(): "A"}
tags["side"] = tags["Team"].fillna("N/A").str.strip().str.lower().map(lambda v: names.get(v, "NA" if v == "n/a" else "?"))

rep("instans: {} | sisi Team: {}".format(len(tags), tags.side.value_counts().to_dict()))
rep.table(tags.code.value_counts().rename("n").to_frame(), "jumlah per kode:")

check("638 instans", len(tags) == 638, str(len(tags)))
check("Team: H 314 / A 288 / N/A 36",
      tags.side.value_counts().to_dict() == {"H": 314, "A": 288, "NA": 36}, str(tags.side.value_counts().to_dict()))
check("59 SET PIECES", int((tags.code.str.upper() == "SET PIECES").sum()) == 59)

is_ko = (tags.code.str.upper() == "SET PIECES") & (tags.get("Type", pd.Series("", index=tags.index))
                                                     .fillna("").str.lower() == "kick off (start)")
for h, ko in KICKOFF.items():
    hit = tags[is_ko & (tags.start_f <= ko) & (tags.end_f >= ko)]
    check("jendela Kick Off (Start) memuat kick-off babak {} ({})".format(h, ko), len(hit) == 1, str(len(hit)))

## 7. ATD broadcast

Opsional (tidak sinkron dengan tactical), tetapi dicek supaya Fase 3.5 tidak menemui kejutan.
Acuan DATA.md: 164.488 baris, 51 slot, bola ada di 50,5% frame. `frame_id`-nya memakai timeline broadcast sendiri,
jadi hanya dilaporkan, tidak dicek.

In [ ]:
bmeta_text = require_asset("atd_meta", "broadcast").read_text(encoding="utf-8")
n_slots_b = len(re.findall(r'playerChannelId="player(\d+)_x"', bmeta_text))

with timer("scan ATD broadcast"):
    b_ids, b_groups, b_ball = scan_atd(require_asset("atd_raw", "broadcast"))
rep("broadcast: baris {} | frame_id {}..{} berurutan {} | grup {} | slot XML {} | bola {:.1f}%".format(
    len(b_ids), b_ids[0], b_ids[-1], bool((np.diff(b_ids) == 1).all()), sorted(b_groups), n_slots_b,
    100 * b_ball.mean()))

check("ATD broadcast 164.488 baris", len(b_ids) == N_FRAMES_BROADCAST, str(len(b_ids)), critical=False)
check("ATD broadcast 51 slot", b_groups == {51} and n_slots_b == 51, "{} / {}".format(sorted(b_groups), n_slots_b),
      critical=False)
check("bola broadcast ≈ 50,5%", abs(100 * b_ball.mean() - 50.5) < 0.1, "{:.2f}".format(100 * b_ball.mean()),
      critical=False)

## 8. Scorer resmi & GT publik (uji asap)

Tiga hal yang membuktikan rantai *label → file → skor* bekerja sama persis dengan milik panitia:

1. **GT publik** sesuai metadatanya: label hanya di frame 11.185–18.684; `H` 3.454 · `A` 2.577 · `D` 1.469.
2. **Baseline resmi** `baseline_smooth50.csv` (ikut di repo resmi) diskor ulang oleh `score_public()` →
   harus sama dengan README: spellF1 **0,3256**, macroF1 **0,6207**.
3. **Bolak-balik submission**: tulis tebakan konstan dengan `write_submission()`, lalu `validate_file()` harus
   lulus dan spellF1-nya harus **0** (EVALUATION.md: tebakan konstan = 0). Penulisan dua kali harus memberi
   hash yang sama (deterministik). File uji ditaruh di `cache/`, bukan `submissions/`.

In [ ]:
gt_lab = pm.read_label_csv(str(GT_PUBLIC))["label"].astype(str).str.strip().str.upper().to_numpy()
covered = np.flatnonzero(gt_lab != "X")
gt_counts = {k: int((gt_lab == k).sum()) for k in "HAD"}
rep("GT publik: {} baris | berlabel {}..{} | {}".format(len(gt_lab), covered[0], covered[-1], gt_counts))
check("GT publik 199.319 baris & frame 11185..18684",
      len(gt_lab) == N_FRAMES and covered[0] == 11185 and covered[-1] == 18684 and len(covered) == 7500)
check("GT publik H 3454 / A 2577 / D 1469", gt_counts == {"H": 3454, "A": 2577, "D": 1469}, str(gt_counts))

base = score_public(OFFICIAL / "evaluation" / "results" / "baseline_smooth50.csv")
rep("baseline_smooth50 (publik): spellF1 {:.4f} | macroF1 {:.4f} | possession {}".format(
    base["spell_f1_mean"], base["macro_f1"], base["n_pred_spells"]))
check("skor ulang baseline = README (0.3256 / 0.6207)",
      round(base["spell_f1_mean"], 4) == 0.3256 and round(base["macro_f1"], 4) == 0.6207)

const = np.where(GRADED, "H", "X")
p1, h1 = write_submission(const, "_uji_konstan", folder=CACHE)
p2, h2 = write_submission(const, "_uji_konstan", folder=CACHE)
ok, summary = validate_file(p1)
sc = score_public(p1)
rep("uji bolak-balik: valid {} | {} | spellF1 {:.4f}".format(ok, summary, sc["spell_f1_mean"]))
check("submission uji lolos validasi resmi", ok, summary.get("reason", ""))
check("tebakan konstan → spellF1 = 0", sc["spell_f1_mean"] == 0.0, str(sc["spell_f1_mean"]))
check("penulisan deterministik (hash sama)", h1 == h2)
p1.unlink()

## 9. Ringkasan

Semua cek **kritis** harus `PASS`. `WARN` dicatat di laporan dan dibahas di bagian keputusan di bawah.

In [ ]:
summary_df = pd.DataFrame(CHECKS)
rep.table(summary_df.set_index("cek")[["status", "keterangan"]], "\nringkasan cek:")
n_fail = int((summary_df.status == "FAIL").sum())
n_warn = int((summary_df.status == "WARN").sum())
rep("\nPASS {} | WARN {} | FAIL {}".format(int((summary_df.status == "PASS").sum()), n_warn, n_fail))

rep.save({"checks": CHECKS, "fingerprint": fingerprint, "inventory": inv.reset_index().to_dict("records"),
          "slots": {"total": N_SLOTS, "home": int(slot_is_home.sum())}, "attack_h1": attack_h1,
          "homography": homo, "ball_present": {"all": n_ball_all, "graded": n_ball_graded}})
assert n_fail == 0, "{} cek kritis gagal — lihat tabel di atas".format(n_fail)

## Ringkasan & keputusan

**Yang dibuktikan notebook ini bila semua cek kritis lulus**

- Data kita identik dengan deskripsi panitia (jumlah baris, frame, slot, kehadiran bola, tag) → angka di DATA.md
  boleh dipakai sebagai acuan.
- Frame selaras: `frame_video = frame_id − 1` untuk ATD, `round(detik × 25)` untuk tag, homography = baris submission.
- Tim `H` = Barcelona dan urutan slot konsisten antara XML dan baris ATD.
- Rantai tulis → validasi → skor memakai kode resmi dan deterministik.

**Keputusan**

| Topik | Keputusan |
|---|---|
| Video tactical | Ada (`BAR-ATH_tactical-002.mp4`, ± 5 GB). **Tidak dibutuhkan** untuk Fase 1–3; disimpan untuk stretch 3.6 (deteksi bola dari video). |
| Nama video tidak standar (`-002`, `-001`) | Tidak diganti; ditangani oleh `asset_path()`. |
| Lingkungan | Run final wajib lewat `run_all` di `.venv` (cek bagian 1 harus PASS semua di run itu). |

**Berikutnya:** `01_preprocess.ipynb` — parsing penuh ATD ke array per frame video (meter, kecepatan, tim per slot).